# Age- and sex-specific cancer prevalence and crude rates in Our Future Health

## Purpose
This notebook extracts self-reported and inpatient EHR-linked cancer diagnoses from Our Future Health (OFH) and computes age- and sex-specific cancer prevalence counts and crude rates (per 100,000 participants) among OFH participants.

## Outputs
- `outputs/intermediate/questionnaire_diag_fields_metadata.csv`  
  Metadata describing questionnaire diagnosis fields used for cancer case definitions.
- `outputs/raw/questionnaire_diag_fields_raw_values_query.sql`  
  SQL query used to extract raw questionnaire diagnosis responses.
- `results.csv`  
  Aggregated age- and sex-specific cancer counts and crude rates (per 100,000) for selected cancer groupings, structured for direct use in Supplementary Table S11.

## Relationship to manuscript
Outputs from this notebook are used to populate **Supplementary Table S11** (*Supplementary Table S11: Comparison of age- and sex-specific self-reported and EHR cancer prevalence and crude rate (per 100 000) among Our Future Health participants and the population of England (2022)*).

## Data and access notes
Analyses use restricted Our Future Health data accessed within the OFH Trusted Research Environment under approved study permissions. The analysis is restricted to participants aged 20 years or older and to registrations prior to June 2024, corresponding to participants registered in England. All outputs are aggregated, non-disclosive summary statistics, consistent with OFH Safe Output requirements.

## Notes
Cancer groupings are derived from self-reported questionnaire diagnosis fields.


## Setup env

In [ ]:
# Import packages
import dxpy
import shlex
import subprocess
import numpy as np
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

# Import phenofhy
import phenofhy

### Initialize Spark

In [ ]:
spark = SparkSession.builder \
    .appName("Phenotype Analysis") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.kryoserializer.buffer.max", "128") \
    .getOrCreate()

### Table S11

#### Extraction

In [ ]:
phenofhy.load.field_list(
    fields=[
        "participant.birth_month",
        "participant.birth_year",
        "participant.registration_month",
        "participant.registration_year",
        "participant.demog_sex_2_1",
        "participant.demog_sex_1_1",
        "participant.pid",
        "questionnaire.diag_1_m",
        "questionnaire.diag_2_m",
        "questionnaire.diag_cancer_1_m",
        "participant.demog_sex_2_1",
        "participant.demog_sex_1_1",
    ],
    output_file="outputs/intermediate/questionnaire_diag_fields_metadata.csv"
)

phenofhy.extract.fields(
    input_file="outputs/intermediate/questionnaire_diag_fields_metadata.csv",
    output_file="outputs/raw/questionnaire_diag_fields_raw_values_query.sql",
    cohort_key="FULL_SAMPLE_ID",
    sql_only=True
)

raw_diag_df = phenofhy.extract.sql_to_pandas(
    "outputs/raw/questionnaire_diag_fields_raw_values_query.sql"
)

p_df = phenofhy.process.participant_fields(
    raw_diag_df, 
    derive='auto',
    min_age=20,   
    age_group_bins = [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, float("inf")],
    age_group_labels = [
        "20–24","25–29","30–34","35–39","40–44",
        "45–49","50–54","55–59","60–64","65–69",
        "70–74","75–79","80–84","85-90", "90+"
    ],
    extra_ranges={"derived.registration_date": (pd.Timestamp.min, pd.Timestamp("2024-06-01"))} # i.e., excl. Scotland clinics)
)

can_df = phenofhy.process.questionnaire_fields(p_df)

#### Analysis

In [ ]:
# compact version: builds item/sex/age rows with counts + rate per 100k (subgroup denominator)
pairs = [("All cancers combined", None), ("Breast", "breast"), ("Colon/rectal", "colon|rectal"),
         ("Prostate", "prostate"), ("Lung or bronchial", "lung|bronchial")]

sex_specs = [([2], "Female"), ([1], "Male"), (list(can_df["derived.sex"].dropna().unique()), "All")]

# age groups ordering (preserve categorical ordering if present)
if pd.api.types.is_categorical_dtype(can_df["derived.age_group"].dtype):
    age_groups = list(can_df["derived.age_group"].cat.categories)
else:
    age_groups = sorted(can_df["derived.age_group"].dropna().unique(), key=lambda x: str(x))

rows = []
for vals, sex_label in sex_specs:
    for ag in age_groups:
        sub = can_df.loc[can_df["derived.sex"].isin(vals) & (can_df["derived.age_group"] == ag)]
        pop = len(sub)
        prev = phenofhy.calculate.prevalence(
            df=sub,
            traits=['questionnaire.diag_1_m', 'questionnaire.diag_2_m', 'questionnaire.diag_cancer_1_m'],
            denominator=("nonmissing"),
        )
        # compute All cancers combined (diag_1_m code 5 OR diag_2_m code 3)
        total_all = (
            prev.loc[(prev['trait'] == 'diag_1_m') & (prev['code'].astype(str) == '5'), 'count'].sum()
            + prev.loc[(prev['trait'] == 'diag_2_m') & (prev['code'].astype(str) == '3'), 'count'].sum()
        )
        cancer_rows = prev.loc[prev['trait'] == 'diag_cancer_1_m'].assign(m=prev['meaning'].astype(str).str.lower())

        # build item rows (All cancers uses total_all; others use pattern match)
        for item_name, pat in pairs:
            cnt = int(total_all) if pat is None else int(cancer_rows[cancer_rows['m'].str.contains(pat, regex=True)]['count'].sum())
            rate = round((cnt / pop * 100_000) if pop > 0 else 0.0, 1)
            rows.append({"item": item_name, "sex": sex_label, "age group": ag, "count": cnt, "rate": rate})

result_df = pd.DataFrame(rows)[["item", "sex", "age group", "count", "rate"]]

# enforce requested ordering
item_order = ["All cancers combined", "Breast", "Colon/rectal", "Lung or bronchial", "Prostate"]
sex_order = ["Female", "Male", "All"]
age_groups = list(dict.fromkeys(age_groups))  # dedupe preserve order

result_df["item"] = pd.Categorical(result_df["item"], categories=item_order, ordered=True)
result_df["sex"] = pd.Categorical(result_df["sex"], categories=sex_order, ordered=True)
result_df["age group"] = pd.Categorical(result_df["age group"], categories=age_groups, ordered=True)

result_df = result_df.sort_values(["item", "sex", "age group"]).reset_index(drop=True)

# Inspect temp results
result_df.to_csv('results.csv')

### EHR analyis

#### Extraction

In [ ]:
ofh_tools.load.field_list(
    input_file="inputs/can_nhse_inpat_diag_icd_fields.csv", 
    output_file="outputs/intermediate/can_nhse_inpat_diag_fields_metadata.csv",
)

ofh_tools.extract.fields(
    input_file="outputs/intermediate/can_nhse_inpat_diag_fields_metadata.csv",
    output_file="outputs/raw/can_nhse_inpat_diag_fields_raw_values_query.sql", 
    cohort_key="FULL_SAMPLE_ID", 
    sql_only=True
)

raw_nhse_inpat_df = ofh_tools.extract.sql_to_pandas(
    "outputs/raw/can_nhse_inpat_diag_fields_raw_values_query.sql"
)

raw_nhse_inpat_df["nhse_eng_inpat.admidate"] = pd.to_datetime(raw_nhse_inpat_df["nhse_eng_inpat.admidate"])

raw_nhse_inpat_df = raw_nhse_inpat_df[
    raw_nhse_inpat_df["nhse_eng_inpat.admidate"] <= pd.Timestamp("2022-12-31")
]

In [ ]:
#### Preprocessing

In [ ]:
# --- ICD matching ---
trait_map = {
    'ehr_cancer':           [f"C{str(i).zfill(2)}" for i in range(0, 98)],
    'ehr_breast':           ['C50'],
    'ehr_colon_rectal':     ['C18', 'C19', 'C20'],
    'ehr_lung_bronchial':   ['C33', 'C34'],
    'ehr_prostate':         ['C61'],
}

# --- ALL CANCER --- 
trait_pids, summary = ofh_tools.icd.match_icd_traits(raw_nhse_inpat_df, trait_map, primary_only=True) # Primary only

# --- FIRST CANCER PER PERSON (for ehr_cancer only) ---
diag_cols = [c for c in raw_nhse_inpat_df.columns if c.startswith('nhse_eng_inpat.diag_4_')]

# flag rows with any cancer code
cancer_mask = raw_nhse_inpat_df[diag_cols].apply(
    lambda col: col.str.startswith('C', na=False)
).any(axis=1)

# keep only cancer rows
df_cancer = raw_nhse_inpat_df.loc[cancer_mask, ["participant.pid", "nhse_eng_inpat.admidate"]]

# get earliest cancer admission per person
first_cancer = (
    df_cancer
    .sort_values("nhse_eng_inpat.admidate")
    .drop_duplicates("participant.pid", keep="first")
)

ehr_cancer_pids= set(first_cancer["participant.pid"])

# --- 1. Filter to rows with any cancer diagnosis (C codes) ---
diag_cols = [c for c in raw_nhse_inpat_df.columns if c.startswith('nhse_eng_inpat.diag_4_')]
cancer_mask = raw_nhse_inpat_df[diag_cols].apply(
    lambda col: col.str.startswith('C', na=False)
).any(axis=1)
cancer_pids = raw_nhse_inpat_df.loc[cancer_mask, 'participant.pid'].unique()

# --- 2. Restrict to those participants (all their rows, for correct age derivation) ---
raw_cancer_df = raw_nhse_inpat_df[raw_nhse_inpat_df['participant.pid'].isin(cancer_pids)]

# get age groups
p_df = ofh_tools.process.participant_fields(
    raw_cancer_df, 
    derive='auto',
    min_age=20,   
    age_group_bins = [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, float("inf")],
    age_group_labels = [
        "20–24","25–29","30–34","35–39","40–44",
        "45–49","50–54","55–59","60–64","65–69",
        "70–74","75–79","80–84","85-90", "90+"
    ],
    extra_ranges={"derived.registration_date": (pd.Timestamp.min, pd.Timestamp("2024-06-01"))} # i.e., excl. Scotland clinics
)

#### Analysis

In [ ]:
# --- STEP 1: participant-level dataset (one row per person) ---
p_unique = (
    p_df
    .drop_duplicates(subset="participant.pid")
    [["participant.pid", "derived.age_group", "derived.sex"]]
    .copy()
)

# --- STEP 2: attach binary flags for each cancer trait ---
# for condition, pids in trait_pids.items():
#     p_unique[condition] = p_unique["participant.pid"].isin(pids).astype(int)

for condition, pids in trait_pids.items():
    if condition == "ehr_cancer":
        p_unique[condition] = p_unique["participant.pid"].isin(ehr_cancer_pids).astype(int)
    else:
        p_unique[condition] = p_unique["participant.pid"].isin(pids).astype(int)

# list of trait columns
trait_cols = list(trait_pids.keys())

# --- STEP 3: keep only valid sex values (1=Male, 2=Female) ---
p_clean = p_unique[p_unique["derived.sex"].isin([1, 2])].copy()

counts_wide = (
    p_clean
    .groupby(["derived.sex", "derived.age_group"])[trait_cols]
    .sum()
    .reset_index()
)

# --- STEP 4: map labels ---
sex_map = {1: "Male", 2: "Female"}
counts_wide["derived.sex"] = counts_wide["derived.sex"].map(sex_map)

# --- STEP 5: add "Persons" ---
counts_persons = (
    counts_wide
    .groupby("derived.age_group")[trait_cols]
    .sum()
    .reset_index()
)
counts_persons["derived.sex"] = "Persons"

counts_wide = pd.concat([counts_wide, counts_persons], ignore_index=True)

# --- STEP 6: remove zero-only rows (KEY FIX) ---
counts_wide = counts_wide[
    counts_wide[trait_cols].sum(axis=1) > 0
]

# --- STEP 7: ordering ---
sex_order = ["Male", "Female", "Persons"]
counts_wide["derived.sex"] = pd.Categorical(
    counts_wide["derived.sex"],
    categories=sex_order,
    ordered=True
)

if pd.api.types.is_categorical_dtype(p_clean["derived.age_group"]):
    counts_wide = counts_wide.sort_values(["derived.sex", "derived.age_group"])
else:
    counts_wide = counts_wide.sort_values(
        ["derived.sex", "derived.age_group"],
        key=lambda col: col.astype(str)
    )

# --- STEP 8: long format ---
counts_long = counts_wide.melt(
    id_vars=["derived.sex", "derived.age_group"],
    value_vars=trait_cols,
    var_name="trait",
    value_name="n"
)

# Inspect temp results
counts_long.to_csv('ehr_results.csv')

#### Crude rate

demo_df = raw_nhse_inpat_df[[
    "participant.pid",
    "participant.birth_year",
    "participant.birth_month",
    "participant.registration_year",
    "participant.registration_month",
    "participant.demog_sex_1_1",
    "participant.demog_sex_2_1"
]].drop_duplicates("participant.pid")

p_full = ofh_tools.process.participant_fields(
    demo_df,
    derive='auto',
    min_age=20,
    age_group_bins=[20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,float("inf")],
    age_group_labels=[
        "20–24","25–29","30–34","35–39","40–44",
        "45–49","50–54","55–59","60–64","65–69",
        "70–74","75–79","80–84","85-90","90+"
    ],
    extra_ranges={"derived.registration_date": (pd.Timestamp.min, pd.Timestamp("2024-06-01"))}
)

denoms = (
    p_full[p_full["derived.sex"].isin([1, 2])]
    .groupby(["derived.sex", "derived.age_group"])["participant.pid"]
    .nunique()
    .reset_index(name="denominator")
)

# map sex labels
sex_map = {1: "Male", 2: "Female"}
denoms["derived.sex"] = denoms["derived.sex"].map(sex_map)

denoms_persons = (
    denoms
    .groupby("derived.age_group")["denominator"]
    .sum()
    .reset_index()
)
denoms_persons["derived.sex"] = "Persons"

denoms = pd.concat([denoms, denoms_persons], ignore_index=True)

rates = counts_long.merge(
    denoms,
    on=["derived.sex", "derived.age_group"],
    how="left"
)

rates["rate_per_100k"] = (rates["n"] / rates["denominator"]) * 100000

### Upload results

In [ ]:
# Upload an entire directory of folders
phenofhy.utils.upload_folders([
    ("phenofhy/", "applets/phenofhy"),
])